# NOAA Global Surface Summary of the Day con KaggleHub y PySpark

Este notebook descarga la base de datos de Kaggle solo si no existe una copia local, la abre con PySpark y genera una exploraciÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â³n inicial para evaluar estructura, faltantes y relevancia para el tema de investigaciÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â³n.


In [2]:
from pathlib import Path
import json
import os

DATASET_SLUG = "noaa/noaa-global-surface-summary-of-the-day"
TOKEN_PATH = Path("Kaggle API Token.txt")
DATASET_PATH_FILE = Path(".noaa_gsod_dataset_path.txt")

# Cambia a True si quieres forzar una descarga nueva.
FORCE_DOWNLOAD = False

# Ajusta estas variables a tu tema de investigaciÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â³n.
RESEARCH_TOPIC = "clima, temperatura, precipitaciÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â³n y eventos meteorolÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â³gicos"
RESEARCH_KEYWORDS = [
    "weather", "climate", "temperature", "temp", "precipitation", "prcp",
    "rain", "snow", "wind", "gust", "pressure", "visibility", "station",
    "date", "year", "month", "day", "tornado", "storm", "fog", "hail"
]
# Windows sin Conda: si instalas winutils.exe y hadoop.dll, coloca aquÃƒÆ’Ã‚Â­ la carpeta raÃƒÆ’Ã‚Â­z de Hadoop.
 # Ejemplo: HADOOP_HOME_PATH = r"C:\hadoop"
HADOOP_HOME_PATH = r"C:\hadoop"
 


In [3]:
def load_kaggle_credentials(raw_text):
    """Acepta JSON de Kaggle, variables KAGGLE_*, usuario=clave, usuario,clave o dos lÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â­neas."""
    try:
        credentials = json.loads(raw_text)
        return credentials["username"], credentials["key"]
    except json.JSONDecodeError:
        pass

    lines = [line.strip() for line in raw_text.splitlines() if line.strip()]
    values = {}
    pairs = []

    for line in lines:
        if "=" in line:
            left, right = line.split("=", 1)
            left = left.strip().strip('"')
            right = right.strip().strip('"')
            values[left.lower()] = right
            pairs.append((left, right))

    if {"kaggle_username", "kaggle_key"}.issubset(values):
        return values["kaggle_username"], values["kaggle_key"]
    if {"username", "key"}.issubset(values):
        return values["username"], values["key"]
    if len(pairs) == 1:
        return pairs[0]
    if len(lines) >= 2:
        return lines[0], lines[1]
    if len(lines) == 1 and "," in lines[0]:
        username, key = lines[0].split(",", 1)
        return username.strip(), key.strip()

    raise ValueError(
        "Kaggle API Token.txt debe contener JSON de Kaggle, "
        "KAGGLE_USERNAME/KAGGLE_KEY, usuario=clave, usuario,clave "
        "o usuario y clave en lÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â­neas separadas."
    )


def configure_kaggle_credentials(token_path=TOKEN_PATH):
    if not token_path.exists():
        raise FileNotFoundError(f"No se encontrÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â³ el token de Kaggle en: {token_path.resolve()}")

    raw_token = token_path.read_text(encoding="utf-8-sig").strip()
    username, key = load_kaggle_credentials(raw_token)
    if not username or not key:
        raise ValueError("El token de Kaggle no contiene username o key.")

    os.environ["KAGGLE_USERNAME"] = username
    os.environ["KAGGLE_KEY"] = key

    kaggle_config_dir = Path.cwd() / ".kaggle"
    kaggle_config_dir.mkdir(exist_ok=True)
    os.environ["KAGGLE_CONFIG_DIR"] = str(kaggle_config_dir)

    kaggle_json_path = kaggle_config_dir / "kaggle.json"
    kaggle_json_path.write_text(
        json.dumps({"username": username, "key": key}),
        encoding="utf-8",
    )
    try:
        kaggle_json_path.chmod(0o600)
    except OSError:
        pass

    return kaggle_json_path

print("Funciones de credenciales listas. El token no se imprime en pantalla.")


Funciones de credenciales listas. El token no se imprime en pantalla.


In [4]:
def path_has_files(path):
    path = Path(path)
    return path.exists() and path.is_dir() and any(item.is_file() for item in path.rglob("*"))


def find_existing_kagglehub_dataset(dataset_slug=DATASET_SLUG):
    if DATASET_PATH_FILE.exists():
        saved_path = Path(DATASET_PATH_FILE.read_text(encoding="utf-8").strip())
        if path_has_files(saved_path):
            return saved_path

    owner, dataset_name = dataset_slug.split("/", 1)
    cache_root = Path.home() / ".cache" / "kagglehub" / "datasets" / owner / dataset_name / "versions"
    if cache_root.exists():
        candidates = [path for path in cache_root.iterdir() if path_has_files(path)]
        if candidates:
            return max(candidates, key=lambda path: path.stat().st_mtime)

    return None


existing_path = None if FORCE_DOWNLOAD else find_existing_kagglehub_dataset()

if existing_path:
    path = str(existing_path)
    print("Dataset ya disponible localmente. No se descargÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â³ de nuevo.")
else:
    configure_kaggle_credentials()
    import kagglehub
    path = kagglehub.dataset_download(DATASET_SLUG)
    print("Dataset descargado desde KaggleHub.")

DATASET_PATH_FILE.write_text(str(path), encoding="utf-8")
print("Path to dataset files:", path)


Dataset ya disponible localmente. No se descargÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â³ de nuevo.
Path to dataset files: C:\Users\isaac\.cache\kagglehub\datasets\noaa\noaa-global-surface-summary-of-the-day\versions\2


In [5]:
import platform
import sys

def configure_windows_hadoop(hadoop_home_path=HADOOP_HOME_PATH):
    """Configura Spark en Windows antes de crear SparkSession."""
    is_windows = platform.system().lower() == "windows"

    # PySpark exige la misma versiÃƒÂ³n menor de Python en driver y workers.
    # En Jupyter, sys.executable apunta al Python exacto del kernel activo.
    os.environ["PYSPARK_PYTHON"] = sys.executable
    os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

    if not is_windows:
        return

    # Workaround para instalaciones standalone: evita usar Hadoop NativeIO cuando sea posible.
    os.environ.setdefault("HADOOP_OPTS", "-Dio.native.lib.available=false -Dhadoop.native.lib=false")

    if hadoop_home_path:
        hadoop_home = Path(hadoop_home_path)
        hadoop_bin = hadoop_home / "bin"
        os.environ["HADOOP_HOME"] = str(hadoop_home)
        os.environ["hadoop.home.dir"] = str(hadoop_home)
        os.environ["PATH"] = str(hadoop_bin) + os.pathsep + os.environ.get("PATH", "")

        missing_files = [
            str(path) for path in [hadoop_bin / "winutils.exe", hadoop_bin / "hadoop.dll"]
            if not path.exists()
        ]
        if missing_files:
            print("Advertencia: HADOOP_HOME estÃƒÂ¡ definido, pero faltan archivos:")
            for missing_file in missing_files:
                print("-", missing_file)
    else:
        print(
            "Windows detectado. Se intentarÃƒÂ¡ iniciar Spark desactivando NativeIO. "
            "Si aparece UnsatisfiedLinkError, instala winutils.exe y hadoop.dll "
            "compatibles y define HADOOP_HOME_PATH en la celda de configuraciÃƒÂ³n."
        )

    print("Python usado por PySpark:", sys.executable)

configure_windows_hadoop()


Python usado por PySpark: c:\Users\isaac\AppData\Local\Programs\Python\Python313\python.exe


In [6]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from functools import reduce
import sys

# Si ya existÃƒÂ­a una sesiÃƒÂ³n creada con otro Python, reinicia el kernel antes de correr desde arriba.
spark = (
    SparkSession.builder
    .appName("noaa-gsod-exploration")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.pyspark.driver.python", sys.executable)
    .config("spark.executorEnv.PYSPARK_PYTHON", sys.executable)
    .config("spark.yarn.appMasterEnv.PYSPARK_PYTHON", sys.executable)
    .config("spark.hadoop.io.native.lib.available", "false")
    .config("spark.hadoop.hadoop.native.lib", "false")
    .config("spark.hadoop.io.nativeio.enabled", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)
print("Driver Python:", sys.executable)
print("Worker Python configurado:", spark.sparkContext.pythonExec)
spark


Spark: 4.1.1
Driver Python: c:\Users\isaac\AppData\Local\Programs\Python\Python313\python.exe
Worker Python configurado: c:\Users\isaac\AppData\Local\Programs\Python\Python313\python.exe


In [7]:
dataset_path = Path(path)
all_files = [file for file in dataset_path.rglob("*") if file.is_file()]

csv_files = [file for file in all_files if file.name.lower().endswith((".csv", ".csv.gz"))]
parquet_files = [file for file in all_files if file.suffix.lower() == ".parquet"]
json_files = [file for file in all_files if file.suffix.lower() == ".json"]

print(f"Archivos encontrados: {len(all_files):,}")
print(f"CSV: {len(csv_files):,} | Parquet: {len(parquet_files):,} | JSON: {len(json_files):,}")

if csv_files:
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .option("recursiveFileLookup", True)
        .option("pathGlobFilter", "*.csv*")
        .csv(str(dataset_path))
    )
    source_format = "csv"
elif parquet_files:
    df = spark.read.parquet(*[str(file) for file in parquet_files])
    source_format = "parquet"
elif json_files:
    df = spark.read.option("recursiveFileLookup", True).json(str(dataset_path))
    source_format = "json"
else:
    raise ValueError("No se encontraron archivos CSV, Parquet o JSON para cargar con PySpark.")

print(f"Formato cargado: {source_format}")
df.printSchema()
df.show(5, truncate=False)


Archivos encontrados: 129
CSV: 2 | Parquet: 0 | JSON: 1
Formato cargado: csv
root
 |-- USAF: string (nullable = true)
 |-- WBAN: string (nullable = true)
 |-- STATION NAME: string (nullable = true)
 |-- CTRY: string (nullable = true)
 |-- STATE: string (nullable = true)
 |-- ICAO: string (nullable = true)
 |-- LAT: double (nullable = true)
 |-- LON: double (nullable = true)
 |-- ELEV(M): double (nullable = true)
 |-- BEGIN: integer (nullable = true)
 |-- END: integer (nullable = true)

+------+-----+------------+----+-----+----+-----+------+-------+--------+--------+
|USAF  |WBAN |STATION NAME|CTRY|STATE|ICAO|LAT  |LON   |ELEV(M)|BEGIN   |END     |
+------+-----+------------+----+-----+----+-----+------+-------+--------+--------+
|007018|99999|WXPOD 7018  |NULL|NULL |NULL|0.0  |0.0   |7018.0 |20110309|20130730|
|007026|99999|WXPOD 7026  |AF  |NULL |NULL|0.0  |0.0   |7026.0 |20120713|20170822|
|007070|99999|WXPOD 7070  |AF  |NULL |NULL|0.0  |0.0   |7070.0 |20140923|20150926|
|008260|999

In [8]:
# Esta celda puede tardar porque cuenta registros sobre una base grande.
# Evitamos spark.createDataFrame(...) con listas locales para no lanzar workers Python auxiliares.
df = df.cache()
num_columns = len(df.columns)
num_records = df.count()

print(f"Número de registros: {num_records:,}")
print(f"Número de columnas: {num_columns:,}")

schema_rows = [
    {
        "columna": field.name,
        "tipo_dato": field.dataType.simpleString(),
        "nullable": field.nullable,
    }
    for field in df.schema.fields
]

try:
    import pandas as pd
    display(pd.DataFrame(schema_rows))
except ImportError:
    for row in schema_rows:
        print(f"{row['columna']} | {row['tipo_dato']} | nullable={row['nullable']}")

numeric_columns = [
    field.name for field in df.schema.fields
    if field.dataType.simpleString() in {"byte", "short", "int", "bigint", "float", "double", "decimal"}
    or field.dataType.simpleString().startswith("decimal")
]

if numeric_columns:
    df.select(numeric_columns).describe().show(truncate=False)
else:
    print("No se detectaron columnas numéricas para describe().")

Número de registros: 29,785
Número de columnas: 11


,columna,tipo_dato,nullable
0,USAF,string,True
1,WBAN,string,True
2,STATION NAME,string,True
3,CTRY,string,True
4,STATE,string,True
5,ICAO,string,True
6,LAT,double,True
7,LON,double,True
8,ELEV(M),double,True
9,BEGIN,int,True


+-------+-----------------+-------------------+------------------+--------------------+-------------------+
|summary|LAT              |LON                |ELEV(M)           |BEGIN               |END                |
+-------+-----------------+-------------------+------------------+--------------------+-------------------+
|count  |28571            |28570              |28477             |29775               |29775              |
|mean   |30.67765573483605|-3.5489847042353584|344.60203321978884|1.9782464456960537E7|2.004973012822838E7|
|stddev |28.80766983223563|87.29042605139709  |590.7465340745981 |234553.38557280094  |193748.37722487454 |
|min    |-90.0            |-179.983           |-999.9            |19010101            |19051231           |
|max    |83.65            |179.75             |8318.0            |20190315            |20190417           |
+-------+-----------------+-------------------+------------------+--------------------+-------------------+



In [9]:
def missing_condition(column_name, data_type):
    column = F.col(f"`{column_name}`")
    condition = column.isNull()
    data_type_name = data_type.simpleString()

    if data_type_name == "string":
        normalized = F.lower(F.trim(column))
        condition = condition | (F.trim(column) == "") | normalized.isin("na", "n/a", "nan", "null", "none", "missing")
    elif data_type_name in {"float", "double"}:
        condition = condition | F.isnan(column)

    return condition


missing_expressions = [
    F.sum(F.when(missing_condition(field.name, field.dataType), 1).otherwise(0)).alias(field.name)
    for field in df.schema.fields
]

missing_counts = df.select(missing_expressions).collect()[0].asDict()
missing_summary_data = [
    {
        "columna": column,
        "valores_faltantes": int(count),
        "porcentaje_faltante": round((int(count) / num_records) * 100, 4) if num_records else 0.0,
    }
    for column, count in missing_counts.items()
]
missing_summary_data = sorted(missing_summary_data, key=lambda row: row["valores_faltantes"], reverse=True)
columns_with_missing_data = [row for row in missing_summary_data if row["valores_faltantes"] > 0]

print("Columnas con valores faltantes:")
try:
    import pandas as pd
    display(pd.DataFrame(columns_with_missing_data))
except ImportError:
    for row in columns_with_missing_data:
        print(f"{row['columna']} | faltantes={row['valores_faltantes']:,} | {row['porcentaje_faltante']}%")

row_missing_condition = reduce(
    lambda left, right: left | right,
    [missing_condition(field.name, field.dataType) for field in df.schema.fields]
)
rows_with_missing = df.filter(row_missing_condition).count()

print(f"Registros con al menos un valor faltante: {rows_with_missing:,}")
print(f"Columnas con al menos un valor faltante: {len(columns_with_missing_data):,}")

Columnas con valores faltantes:


,columna,valores_faltantes,porcentaje_faltante
0,STATE,23042,77.3611
1,ICAO,18894,63.4346
2,ELEV(M),1308,4.3915
3,LON,1215,4.0792
4,LAT,1214,4.0759
5,CTRY,947,3.1795
6,STATION NAME,800,2.6859
7,BEGIN,10,0.0336
8,END,10,0.0336
9,WBAN,7,0.0235


Registros con al menos un valor faltante: 24,667
Columnas con al menos un valor faltante: 10


In [10]:
keywords = [keyword.lower() for keyword in RESEARCH_KEYWORDS]
lower_columns = {column: column.lower() for column in df.columns}

matched_columns = [
    column for column, lower_column in lower_columns.items()
    if any(keyword in lower_column for keyword in keywords)
]

print("Tema de investigaciÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â³n:", RESEARCH_TOPIC)
print("Columnas que coinciden con palabras clave:")
for column in matched_columns:
    print("-", column)

if matched_columns:
    print(
        "\nConclusiÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â³n preliminar: la base sÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â­ contiene variables potencialmente relevantes "
        "para el tema indicado, porque hay columnas asociadas a fecha, estaciÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â³n "
        "y/o variables meteorolÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â³gicas."
    )
else:
    print(
        "\nConclusiÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â³n preliminar: no se encontraron coincidencias directas en los nombres "
        "de columnas. Ajusta RESEARCH_KEYWORDS o revisa una muestra de registros."
    )

string_columns = [field.name for field in df.schema.fields if field.dataType.simpleString() == "string"]
sample_for_relevance = df.select(string_columns[:10]).limit(1000) if string_columns else None

if sample_for_relevance is not None and string_columns:
    print("\nMuestra de columnas de texto para inspecciÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â³n cualitativa:")
    sample_for_relevance.show(10, truncate=False)


Tema de investigaciÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â³n: clima, temperatura, precipitaciÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â³n y eventos meteorolÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â³gicos
Columnas que coinciden con palabras clave:
- STATION NAME

ConclusiÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â³n preliminar: la base sÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â­ contiene variables potencialmente relevantes para el tema indicado, porque hay columnas asociadas a fecha, estaciÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â³n y/o variables meteorolÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â³gicas.

Muestra de columnas de texto para inspecciÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â³n cualitativa:
+------+-----+------------+----+-----+----+
|USAF  |WBAN |STATION NAME|CTRY|STATE|ICAO|
+------+-----+------------+----+-----+----+
|007018|99999|WXPOD 7018  |NULL|NULL |NULL|
|007026|99999|WXPOD 7026  |AF  |NULL |NULL|
|007070|99999|WXPOD 7070  |AF  |NULL |NULL|
|008260|99999|WXPOD8270   |NULL|NULL |NULL|
|008268|99999|WXPOD8278   |AF  |NULL |NULL|
|008307|99999|WXPOD 8318  |AF  |NULL |NULL|
|008411|99999|XM20        |NULL|NULL |NULL|
|008414|99999|XM18        |NULL|NULL |NULL|
